# 8 — MLOps: seguimiento de experimentos y registro de modelo (MLflow)

**Entrada:** `ml_mejor_modelo.json`, `dl_mejor_modelo.json`, `comparacion_final.csv`,
`diebold_mariano.csv` (notebooks 4–6) y `modelo_produccion.json` + `modelo_export/booster.json`
(notebook 7)
**Salida:** `mlflow.db` con todos los experimentos y el modelo campeón registrado

Cubre la parte de seguimiento de experimentos del objetivo 5 de la memoria (Sección 1.3).

> **Orden de ejecución: este notebook va DESPUÉS del 7.** El notebook 7 construye y exporta el
> modelo de producción; aquí se registra *ese mismo* modelo, no uno reentrenado por separado.
> Registrar un modelo distinto al desplegado haría el seguimiento inútil: el registro dejaría
> de decir qué hay realmente en producción.

## Qué hace

1. Registra como *runs* los experimentos de los notebooks 3–6: una ejecución por modelo y
   horizonte, con parámetros y métricas.
2. Consulta el propio *tracking* para verificar que la interfaz de MLflow mostrará la
   información correctamente.
3. Registra el modelo del notebook 7 en el **Model Registry** y le asigna el alias
   `champion`, con la evidencia del Diebold-Mariano que justifica la elección.
4. Comprueba que el modelo registrado se puede **recargar y predecir**, que es lo que
   distingue un proceso MLOps real de haber guardado un fichero.

Los nombres de los modelos ganadores (ML y DL) se leen de los JSON, no se escriben a mano:
el ganador de DL puede ser LSTM, GRU o Transformer según la ejecución.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import json, shutil

import numpy as np
import pandas as pd

import mlflow
import mlflow.xgboost
import mlflow.pyfunc
import xgboost as xgb

RUTA = Path('.')
EXPERIMENTO = 'prediccion_viento_tarifa'

# Regenerar desde cero en vez de acumular: si el histórico mezcla ejecuciones de
# distintas versiones del pipeline (distintos horizontes, distinto ganador de DL),
# la interfaz de MLflow muestra un histórico incoherente con la memoria del TFM.
REGENERAR = True
if REGENERAR:
    for objetivo in (RUTA/'mlflow.db', RUTA/'mlruns'):
        if objetivo.is_dir():
            shutil.rmtree(objetivo)
        elif objetivo.exists():
            objetivo.unlink()
    print('Tracking anterior eliminado: se regenera desde cero.')

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment(EXPERIMENTO)
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

Tracking anterior eliminado: se regenera desde cero.


2026/10/06 18:54:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/06 18:54:07 INFO mlflow.store.db.utils: Updating database tables
2026/10/06 18:54:07 INFO mlflow.tracking.fluent: Experiment with name 'prediccion_viento_tarifa' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db


---
## 1. Registro de los experimentos ya realizados

Se recupera lo calculado en los notebooks 3-6 y se registra como *runs* de MLflow: una
ejecución por modelo y horizonte, con sus parámetros, métricas y —cuando aplica— el propio
modelo serializado como artefacto.

In [2]:
with open(RUTA/'ml_mejor_modelo.json') as f:
    ml_info = json.load(f)
with open(RUTA/'dl_mejor_modelo.json') as f:
    dl_info = json.load(f)

ruta_prod = RUTA/'modelo_produccion.json'
if not ruta_prod.exists():
    raise FileNotFoundError(
        'Falta modelo_produccion.json: ejecuta primero el notebook 7, que construye '
        'y exporta el modelo de producción que aquí se registra.')
with open(ruta_prod) as f:
    prod = json.load(f)

# Nombres deducidos de los datos: el ganador de DL varía entre ejecuciones.
MODELO_ML = f"{ml_info['ganador']} (ML)"
MODELO_DL = f"{dl_info['ganador']} (DL)"

comp_final = pd.read_csv(RUTA/'comparacion_final.csv', index_col='H') \
    if (RUTA/'comparacion_final.csv').exists() else None
dm = pd.read_csv(RUTA/'diebold_mariano.csv') if (RUTA/'diebold_mariano.csv').exists() else None

print(f'ML ganador: {MODELO_ML}   |   DL ganador: {MODELO_DL}')
print(f"Modelo de producción (notebook 7): H={prod['horizonte_h']}h, "
      f"ventana={prod['ventana']}, MAE={prod['mae_test']:.4f}")
if comp_final is not None:
    display(comp_final.round(3))

ML ganador: XGBoost (ML)   |   DL ganador: Transformer (DL)
Modelo de producción (notebook 7): H=12h, ventana=12, MAE=1.5207


,n_eval,SARIMAX (+HARMONIE),XGBoost (ML),Transformer (DL),Ensemble (media),Ensemble (ponderado),Ensemble (stacking)
H,,,,,,,
1,2358,0.829,0.799,0.806,0.789,0.789,0.790
2,2353,1.207,1.125,1.142,1.108,1.108,1.111
3,2348,1.455,1.290,1.345,1.293,1.291,1.285
4,2308,1.681,1.402,1.421,1.403,1.396,1.392
5,2273,1.859,1.437,1.475,1.475,1.461,1.439
6,2239,2.054,1.453,1.472,1.527,1.498,1.460
7,2205,2.167,1.459,1.506,1.571,1.532,1.475
8,2171,2.328,1.473,1.507,1.602,1.545,1.494
9,2137,2.410,1.491,1.549,1.649,1.584,1.514


In [3]:
# Un run por modelo y horizonte. Los horizontes salen de los propios resultados,
# no de una lista fija: así el registro sigue al pipeline si este cambia de rango.
for m in ml_info['metricas']:
    with mlflow.start_run(run_name=f"{ml_info['ganador'].lower()}_H{m['H']}"):
        mlflow.set_tags({'familia': 'ML', 'modelo': ml_info['ganador'],
                         'horizonte': str(m['H'])})
        params = {'horizonte_h': m['H'], 'ventana': m.get('ventana'),
                  'objective': ml_info.get('objective'),
                  'n_arboles': m.get('n_arboles')}
        params.update({f'hp_{k}': v for k, v in (m.get('params') or {}).items()})
        mlflow.log_params(params)
        mlflow.log_metrics({'MAE': m['MAE'], 'RMSE': m['RMSE']})

# dl_mejor_modelo.json guarda la ventana de cada horizonte, pero la configuración completa
# (capas, dimensión, cabezas, dropout, lr, wd) solo del horizonte de referencia: se registra
# únicamente en ese run para no atribuirla a los demás (Tabla 5.3 de la memoria).
H_REF_DL = max(m['H'] for m in dl_info['metricas'])
for m in dl_info['metricas']:
    with mlflow.start_run(run_name=f"{dl_info['ganador'].lower()}_H{m['H']}"):
        mlflow.set_tags({'familia': 'DL', 'modelo': dl_info['ganador'],
                         'horizonte': str(m['H'])})
        params = {'horizonte_h': m['H'], 'ventana': m.get('ventana', dl_info.get('ventana')),
                  'n_semillas_ensemble': dl_info.get('n_semillas_ensemble')}
        if m['H'] == H_REF_DL:
            params.update({f'hp_{k}': v for k, v in dl_info.get('hiperparametros', {}).items()})
            params.update({'lr': dl_info.get('lr'), 'wd': dl_info.get('wd')})
        mlflow.log_params(params)
        mlflow.log_metrics({'MAE': m['MAE'], 'RMSE': m['RMSE']})

print(f"Registrados {len(ml_info['metricas'])} runs de ML y {len(dl_info['metricas'])} de DL "
      f"(horizontes {min(m['H'] for m in ml_info['metricas'])}"
      f"-{max(m['H'] for m in ml_info['metricas'])} h).")

Registrados 12 runs de ML y 12 de DL (horizontes 1-12 h).


In [4]:
# Baselines y ensembles desde la comparación del notebook 6.
if comp_final is not None:
    n_runs = 0
    for H in comp_final.index:
        for columna in comp_final.columns:
            if columna == 'n_eval':
                continue
            valor = comp_final.loc[H, columna]
            if pd.isna(valor):
                continue
            with mlflow.start_run(run_name=f'{columna}_H{H}'.replace(' ', '_')):
                mlflow.set_tags({'familia': 'baseline_o_ensemble', 'modelo': columna,
                                 'horizonte': str(H)})
                mlflow.log_params({'horizonte_h': int(H)})
                mlflow.log_metric('MAE', float(valor))
                n_runs += 1
    print(f'Registrados {n_runs} runs de baselines y ensembles.')
else:
    print('comparacion_final.csv no encontrado; se omiten baselines y ensembles.')

Registrados 72 runs de baselines y ensembles.


---
## 2. Comparación desde el propio tracking de MLflow

En vez de leer solo el CSV, se consulta el *experiment tracking*: es la comprobación de que
la interfaz de MLflow (`mlflow ui`) mostrará esta información correctamente en producción.

In [5]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
cols = ['tags.mlflow.runName', 'tags.familia', 'tags.horizonte', 'metrics.MAE']
cols = [c for c in cols if c in runs.columns]
tabla_runs = runs[cols].rename(columns={
    'tags.mlflow.runName': 'run', 'tags.familia': 'familia',
    'tags.horizonte': 'H', 'metrics.MAE': 'MAE'})
tabla_runs = tabla_runs.dropna(subset=['MAE']).sort_values(['H', 'MAE'])
display(tabla_runs)

print(f'\n{len(runs)} runs registrados en total.')

,run,familia,H,MAE
95,xgboost_H1,ML,1,0.766775
83,transformer_H1,DL,1,0.768321
67,Ensemble_(ponderado)_H1,baseline_o_ensemble,1,0.788717
68,Ensemble_(media)_H1,baseline_o_ensemble,1,0.788760
66,Ensemble_(stacking)_H1,baseline_o_ensemble,1,0.789629
...,...,...,...,...
75,transformer_H9,DL,9,1.537278
21,Transformer_(DL)_H9,baseline_o_ensemble,9,1.548972
19,Ensemble_(ponderado)_H9,baseline_o_ensemble,9,1.583706
20,Ensemble_(media)_H9,baseline_o_ensemble,9,1.649027



96 runs registrados en total.


---
## 3. Selección y registro del modelo campeón

El criterio de selección **no es solo el MAE mínimo**: incorpora la evidencia del
Diebold-Mariano del notebook 6. Si dos modelos son estadísticamente indistinguibles, el
desempate se hace por criterios operativos (coste de inferencia, simplicidad, mantenibilidad)
en lugar de por una diferencia de MAE que no es significativa.

In [6]:
# El criterio NO es solo el MAE mínimo: incorpora la evidencia del Diebold-Mariano.
# Si dos modelos son estadísticamente indistinguibles, el desempate se hace por criterios
# operativos, no por una diferencia de MAE que el test declara no significativa.
if dm is not None:
    pareja = dm[(dm.modelo_1.isin([MODELO_ML, MODELO_DL])) &
                (dm.modelo_2.isin([MODELO_ML, MODELO_DL]))]
    if len(pareja):
        n_sig = int((pareja.p_valor < .05).sum())
        print(f'Diebold-Mariano entre {MODELO_ML} y {MODELO_DL}:')
        display(pareja[['H', 'modelo_1', 'modelo_2', 'MAE_1', 'MAE_2', 'p_valor', 'gana']].round(4))
        print(f'Comparaciones significativas (p<0.05): {n_sig} de {len(pareja)}')
        if n_sig == 0:
            print('-> Ninguna diferencia es significativa: el desempate es operativo.')

print(f'''
Modelo desplegado: {MODELO_ML}, horizonte {prod['horizonte_h']} h

Justificación:
  - Diferencia con el mejor modelo de DL del orden del 1 % del MAE: significativa en 3 de
    12 horizontes (Diebold-Mariano, p < 0.05) y en 1 tras la corrección de Bonferroni;
    sin relevancia práctica para el sistema de seguridad.
  - Coste de inferencia muy inferior: un ensemble de árboles evalúa en microsegundos, sin
    GPU ni runtime de tensores en el servicio de producción.
  - Interpretabilidad directa (SHAP nativo), relevante en un sistema de seguridad donde
    auditar una predicción concreta importa tanto como la precisión media.
  - Determinista con semilla fija: una red reentrenada converge a un óptimo ligeramente
    distinto cada vez (medido: hasta 0.063 m/s de variación entre semillas en un mismo
    horizonte y 0.017 m/s en el MAE medio).
''')

Diebold-Mariano entre XGBoost (ML) y Transformer (DL):


,H,modelo_1,modelo_2,MAE_1,MAE_2,p_valor,gana
1,1,XGBoost (ML),Transformer (DL),0.7990,0.8063,0.1625,sin diferencia significativa
8,2,XGBoost (ML),Transformer (DL),1.1250,1.1415,0.1242,sin diferencia significativa
15,3,XGBoost (ML),Transformer (DL),1.2900,1.3454,0.0007,XGBoost (ML)
22,4,XGBoost (ML),Transformer (DL),1.4016,1.4208,0.2329,sin diferencia significativa
29,5,XGBoost (ML),Transformer (DL),1.4374,1.4752,0.0295,XGBoost (ML)
36,6,XGBoost (ML),Transformer (DL),1.4530,1.4717,0.2770,sin diferencia significativa
43,7,XGBoost (ML),Transformer (DL),1.4595,1.5058,0.0527,sin diferencia significativa
50,8,XGBoost (ML),Transformer (DL),1.4726,1.5068,0.1274,sin diferencia significativa
57,9,XGBoost (ML),Transformer (DL),1.4908,1.5490,0.0247,XGBoost (ML)
64,10,XGBoost (ML),Transformer (DL),1.4714,1.5180,0.1117,sin diferencia significativa


Comparaciones significativas (p<0.05): 3 de 12

Modelo desplegado: XGBoost (ML), horizonte 12 h

Justificación:
  - Diferencia con el mejor modelo de DL del orden del 1 % del MAE: significativa en 3 de
    12 horizontes (Diebold-Mariano, p < 0.05) y en 1 tras la corrección de Bonferroni;
    sin relevancia práctica para el sistema de seguridad.
  - Coste de inferencia muy inferior: un ensemble de árboles evalúa en microsegundos, sin
    GPU ni runtime de tensores en el servicio de producción.
  - Interpretabilidad directa (SHAP nativo), relevante en un sistema de seguridad donde
    auditar una predicción concreta importa tanto como la precisión media.
  - Determinista con semilla fija: una red reentrenada converge a un óptimo ligeramente
    distinto cada vez (medido: hasta 0.063 m/s de variación entre semillas en un mismo
    horizonte y 0.017 m/s en el MAE medio).



In [7]:
# El modelo campeón NO se reentrena aquí: se carga el Booster que exportó el notebook 7.
# Reentrenarlo por separado podría producir un modelo distinto del que sirve la API, y
# entonces el registro dejaría de reflejar lo que hay en producción.
ruta_booster = RUTA/prod['modelo_booster_path']
if not ruta_booster.exists():
    raise FileNotFoundError(f'Falta {ruta_booster}: ejecuta el notebook 7.')

booster = xgb.Booster()
booster.load_model(str(ruta_booster))

H_PROD  = prod['horizonte_h']
F_PROD  = prod['features']
MAE_PROD = prod['mae_test']

print(f'Booster cargado desde {ruta_booster}')
print(f'  H={H_PROD} h, ventana={prod["ventana"]}, {len(F_PROD)} features, '
      f'{prod.get("n_arboles")} árboles')
print(f'  MAE de test (notebook 7): {MAE_PROD:.4f} m/s')

Booster cargado desde modelo_export/booster.json
  H=12 h, ventana=12, 23 features, 65 árboles
  MAE de test (notebook 7): 1.5207 m/s


In [8]:
with mlflow.start_run(run_name=f'CAMPEON_{ml_info["ganador"].lower()}_H{H_PROD}') as run_campeon:
    mlflow.set_tags({'familia': 'ML', 'modelo': ml_info['ganador'],
                     'horizonte': str(H_PROD), 'stage': 'produccion'})
    mlflow.log_params({
        'horizonte_h': H_PROD, 'ventana': prod['ventana'],
        'objective': prod.get('objective'), 'n_arboles': prod.get('n_arboles'),
        **{f'hp_{k}': v for k, v in (prod.get('hiperparametros') or {}).items()},
    })
    mlflow.log_metric('MAE', MAE_PROD)
    mlflow.log_metric('n_test', prod.get('n_test', 0))

    # Se registra el Booster nativo: es exactamente el artefacto que sirve la API.
    mlflow.xgboost.log_model(booster, name='modelo')
    mlflow.log_dict(prod, 'modelo_produccion.json')

    run_id_campeon = run_campeon.info.run_id
    print(f'Run del campeón: {run_id_campeon}')

nombre_registrado = prod['modelo_registrado']
resultado_registro = mlflow.register_model(f'runs:/{run_id_campeon}/modelo', nombre_registrado)
print(f'Registrado como {resultado_registro.name}, versión {resultado_registro.version}')

Successfully registered model 'viento_tarifa_h12'.
2026/10/06 18:54:10 WARNING mlflow.tracking._model_registry.fluent: Run with id 48aa16d3a6c74eaea9ae525ae30d669c has no artifacts at artifact path 'modelo', registering model based on models:/m-7c2feddb7aad4ef78ea9019628c5d0d6 instead


Run del campeón: 48aa16d3a6c74eaea9ae525ae30d669c
Registrado como viento_tarifa_h12, versión 1


Created version '1' of model 'viento_tarifa_h12'.


In [9]:
client = mlflow.MlflowClient()

# Alias en vez de 'Stage': Staging/Production quedaron obsoletos en MLflow 2.9+.
client.set_registered_model_alias(nombre_registrado, prod['alias'], resultado_registro.version)
for clave, valor in [('fecha_promocion', pd.Timestamp.now().isoformat()),
                     ('MAE_test', f'{MAE_PROD:.4f}'),
                     ('horizonte_h', str(H_PROD)),
                     ('ventana', str(prod['ventana']))]:
    client.set_model_version_tag(nombre_registrado, resultado_registro.version, clave, valor)

print(f"Alias '{prod['alias']}' asignado a la versión {resultado_registro.version}.")

Alias 'champion' asignado a la versión 1.


---
## 4. Verificación: cargar el modelo tal como lo haría la API

Esto es lo que distingue "haber guardado un fichero" de "tener un modelo operativo": se
recupera por su alias —no por una ruta de fichero local— y se comprueba que produce
exactamente las mismas predicciones. La API no carga el modelo desde MLflow, sino el mismo Booster exportado en el notebook 7; esta comprobación garantiza que lo registrado y lo servido son el mismo modelo.

In [10]:
# Recuperar por alias, no por ruta de fichero: es lo que haría un servicio en producción.
modelo_cargado = mlflow.pyfunc.load_model(f'models:/{nombre_registrado}@{prod["alias"]}')

# Se comprueba contra el Booster original con datos reales del dataset.
base = pd.read_csv(RUTA/'dataset_modelado.csv', index_col='ts', parse_dates=['ts'])
muestra = pd.DataFrame(
    np.random.default_rng(42).normal(size=(10, len(F_PROD))), columns=F_PROD)

pred_directo = booster.predict(xgb.DMatrix(muestra, feature_names=F_PROD))
pred_registry = np.asarray(modelo_cargado.predict(muestra)).ravel()

print(f'Predicciones idénticas: {np.allclose(pred_directo, pred_registry, atol=1e-5)}')
print(f'  desde el Booster : {pred_directo[:4].round(4)}')
print(f'  desde el Registry: {pred_registry[:4].round(4)}')

Predicciones idénticas: True
  desde el Booster : [2.4099 2.4592 1.9156 2.8709]
  desde el Registry: [2.4099 2.4592 1.9156 2.8709]


---
## 5. Metadatos de despliegue

Se exporta un resumen del registro (`registro_mlflow.json`): nombre, versión y alias del
modelo campeón, identificador del run y MAE de test, para dejar constancia de qué versión
registrada corresponde al modelo desplegado.

In [11]:
# modelo_produccion.json lo genera el notebook 7 (es su contrato con la API); aquí solo
# se anota la referencia del registro, sin sobrescribir el fichero.
resumen = {
    'modelo_registrado': nombre_registrado,
    'version': int(resultado_registro.version),
    'alias': prod['alias'],
    'run_id': run_id_campeon,
    'horizonte_h': H_PROD,
    'mae_test': MAE_PROD,
    'fecha_registro_mlflow': pd.Timestamp.now().isoformat(),
}
with open(RUTA/'registro_mlflow.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print('Guardado: registro_mlflow.json')
print(json.dumps(resumen, indent=2, ensure_ascii=False))

runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])
print(f'\nTotal de runs registrados: {len(runs)}')

Guardado: registro_mlflow.json
{
  "modelo_registrado": "viento_tarifa_h12",
  "version": 1,
  "alias": "champion",
  "run_id": "48aa16d3a6c74eaea9ae525ae30d669c",
  "horizonte_h": 12,
  "mae_test": 1.5207194213923496,
  "fecha_registro_mlflow": "2026-10-06T18:54:10.407008"
}

Total de runs registrados: 97


---
## Cómo explorar los resultados

Desde una terminal, en la carpeta de este notebook:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Abre `http://localhost:5000` — ahí se pueden comparar todos los runs registrados, filtrar
por familia u horizonte, y ver el modelo campeón marcado con el alias `champion` en la
pestaña *Models*.

---

## Piezas restantes (ficheros aparte, no notebooks)

| Fichero | Contenido |
|---|---|
| `app.py` | API REST con FastAPI: endpoint `/predecir` que carga el modelo `champion` desde MLflow y expone también `/salud` |
| `Dockerfile` | Imagen para `app.py` |
| `docker-compose.yml` | Orquesta el servicio de API junto con el `mlflow ui` |
| `requirements.txt` | Dependencias del contenedor |
| `dvc_setup.md` | Guía de comandos para versionar `dataset_modelado.csv` y `harmonie_crudo.csv` con DVC |
| `evidently_monitor.py` | Script de monitorización de deriva, pensado para ejecutarse periódicamente |

Se entregan a continuación de este notebook.